# Langfuse Adapter Demo

This notebook demonstrates how to use the **HallucinationEvaluator** with the sklearn compatible **EPR** and **WEPR** detectors to score LLM traces in Langfuse.

## Prerequisites

* Install the adapter dependencies:
    ```bash
    uv pip install artefactual[adapters]
    ```

* Create a free LangFuse project at [cloud.langfuse.com](https://cloud.langfuse.com), then go to **Settings → API Keys**.

* Set the following environment variables:

    | Variable | Required | Purpose |
    | --- | --- | --- |
    | `OPENAI_API_KEY` | yes | Credential for the endpoint |
    | `OPENAI_BASE_URL` | no | Any OpenAI-compatible endpoint |
    | `OPENAI_MODEL` | no | Model to generate with |
    | `TOP_LOGPROBS` | no | Ranks to request per token (default 15) |
    | `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, `LANGFUSE_HOST` | yes | Langfuse project |

    The endpoint must return `logprobs` with `top_logprobs`, otherwise there is nothing to score.

    `TOP_LOGPROBS` is passed to the detectors as `k`. Every shipped WEPR weights file is
    calibrated at 15 ranks, so raising it will make `wepr()` raise — deliberately, since the
    coefficient vector is fixed at its calibration rank count. Either leave it at 15 or
    supply weights calibrated at the value you choose. `epr()` is unaffected: an EPR
    calibration is a single coefficient, and `k` only governs how the rank axis is aligned. The defaults below target the HuggingFace router, where `OPENAI_API_KEY` is a token from your HuggingFace account.

In [ ]:
# Colab, or any other kernel that is not this repository's environment: install what this
# notebook needs.
# A checkout that ran `uv sync --extra adapters` has both already, so nothing below runs
# there.
#
# uv rather than pip, because pip is the slow half of the wait: installing this package
# into an empty environment measured 17 s under pip, against 3 s to pip-install uv plus 1 s
# for uv to do the same work. On Colab, where most of the dependency tree is already
# present, the gap is smaller.
#
# Plain Python rather than the `!pip` and `%pip` magics, so the cell stays valid Python:
# the tests that run these notebooks compile the code cells, and so do the linters.
import importlib
import importlib.metadata
import subprocess
import sys
from pathlib import Path


def missing(distribution):
    """Whether `distribution` is installed in the interpreter running this kernel.

    Asked of the installed distribution rather than of an import, because a bare directory
    named `artefactual` -- which is what cloning this repository beside the notebook leaves
    behind -- is an empty namespace package: `find_spec` finds it and `import artefactual`
    succeeds, so both would report the package present and skip the install. Only the
    metadata distinguishes a directory from a package.
    """
    try:
        importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return True
    return False


def shadowed(name):
    """Whether a directory beside the notebook hides an installed package of that name.

    The working directory comes first on `sys.path`, so such a directory wins over
    anything installed and the import fails on a submodule, several cells from the cause.
    """
    return Path(name).is_dir() and not Path(name, "__init__.py").exists()


def install(*packages):
    """Install into the interpreter running this kernel, showing what went wrong if it does.

    `-q` and no captured output is how an install failure becomes a bare
    `CalledProcessError` with the resolver's explanation nowhere on screen.
    """
    bootstrap = subprocess.run([sys.executable, "-m", "pip", "install", "-qU", "uv"], capture_output=True, text=True)
    resolve = [sys.executable, "-m", "uv", "pip", "install", "--python", sys.executable, "-q", *packages]
    done = bootstrap if bootstrap.returncode else subprocess.run(resolve, capture_output=True, text=True)
    if done.returncode:
        print(done.stderr or done.stdout)
        done.check_returncode()
    # The kernel started before these files existed, so the import machinery has a cached
    # listing of a directory that did not contain them.
    importlib.invalidate_caches()


if missing("artefactual") or missing("langfuse"):
    install("artefactual[adapters]")

assert not shadowed("artefactual"), (
    "a directory named 'artefactual' beside this notebook is hiding the installed "
    "package; rename it, or run this notebook from somewhere else"
)

## Run a generation and send it to Langfuse

In [ ]:
import os
import time

from langfuse import get_client, observe
from langfuse.openai import OpenAI
from openai.types.chat import ChatCompletion

from artefactual.adapters.langfuse.evaluator import HallucinationEvaluator
from artefactual.scoring.base_detector import epr, wepr

# --- Configuration ---------------------------------------------------------
OPENAI_BASE_URL = os.environ.get("OPENAI_BASE_URL", "https://router.huggingface.co/v1")
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "Qwen/Qwen3-Coder-30B-A3B-Instruct")
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]  # no default: it is a credential
TOP_LOGPROBS = int(os.environ.get("TOP_LOGPROBS", "15"))

# Both factories resolve a registry model name; pass a path instead to use your
# own calibration. K must match the rank count requested above.
DETECTOR_MODEL = "mistralai/Ministral-8B-Instruct-2410"
# ---------------------------------------------------------------------------

client = OpenAI(base_url=OPENAI_BASE_URL, api_key=OPENAI_API_KEY)


@observe()
def run_generation() -> ChatCompletion:
    return client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "What is the capital of France?"},
        ],
        logprobs=True,
        top_logprobs=TOP_LOGPROBS,
    )


print("Generated message:", run_generation().choices[0].message.content)

langfuse = get_client()
langfuse.flush()

print("Waiting for Langfuse server to index the logprobs of the trace...")
time.sleep(3.0)

traces_to_evaluate = langfuse.api.trace.list(limit=1).data

## Score traces with EPR

A detector always needs calibration weights: pass a path to a weights file, or a registered model name. Calling `epr()` with no argument raises `UncalibratedModelError`.

In [ ]:
evaluator = HallucinationEvaluator(
    name="EPR",
    langfuse_client=langfuse,
    detector=epr(DETECTOR_MODEL, k=TOP_LOGPROBS),
)

for trace in traces_to_evaluate:
    score = evaluator.score_trace(trace.id)
    print(f"EPR Scored Trace : {trace.id} → {score}")

langfuse.flush()

## Score traces with WEPR

In [ ]:
evaluator = HallucinationEvaluator(
    name="WEPR",
    langfuse_client=langfuse,
    detector=wepr(DETECTOR_MODEL, k=TOP_LOGPROBS),
)

for trace in traces_to_evaluate:
    score = evaluator.score_trace(trace.id)
    print(f"WEPR Scored Trace : {trace.id} → {score}")

langfuse.flush()